In [ ]:
# Run this cell once when starting on notebook.
# On Google Colab, wait for the Google Drive permission prompts before proceeding
import matplotlib.pyplot as plt
import os
import sys
!pip install datascience
from datascience import *
import numpy as np
import math


from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/data
!mkdir -p /content/data/midterm
!gdown --folder https://drive.google.com/drive/folders/1XifjqyxFHvjK6nOUi4doYsG12hGPhOcI?usp=drive_link -O /content/data
!gdown --folder https://drive.google.com/drive/folders/1JsmzNmbTFJ4RYZp2N4L55p2smMNnxcaL?usp=drive_link -O /content/data/midterm
DATA_FOLDER="/content/data/"
!ls -l /content/data


%matplotlib inline


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.1/725.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.8 MB/s eta 0:00:00
Mounted at /content/drive
Retrieving folder contents
Processing file 1hOMU36QJH-8E1xMqZqick7tdYPtRqXWc codes.csv
Processing file 1-pQmPDnV44eDEVIOm_dglgktbO1ng7pe data_science_salaries_raw.csv
Processing file 1Cl56p5ygZPsG_TdGagMpd2wxMAnsNUN0 ucsb-f25-courses.csv
Processing file 1A6TJBGq7Fg5eO_4XrHgrRRM27RRxB6w7 ucsb-m25-courses.csv
Processing file 1KqFgPgttI11ADhBdGWipQ4RaCyM7S8iZ ucsb-s25-courses.csv
Processing file 1DHBdvECi_-cQtiJ1FZl4v_JfLgAMrauE world_happiness_2015.csv
Processing file 10lU_ehl4CmZqXZ8Ow4tz_G4zHvB9kkwV world_happiness_2019.csv
Processing file 17-CMnZGvqhCNc4_dzlS86njmmPWl_c2g world_university_rankings.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1hOMU36QJH-8E1xMqZqick7tdYPtRqXWc
To: /content/da

# Data Cleaning, Part 0





Real-world datasets are not always *clean*, meaning they may contain some data that's either missing or just plain *wrong*.

The process of taking a data set and removing invalid values is called *data cleaning*.  

It's a process that has to be done thoughtfully, because if not done carefully, it can introduce bias into the data.   But *not* doing it can *also* introduce bias.

In choosing how to clean your data, it's important to know what your purpose is in analyzing the data.

Examples:

* Suppose we are exploring class sizes at UCSB.  If our data set contains courses that are not "regular classroom courses", e.g. individual research classes taught 1-on-1, including those in the data may throw off any kind of averages, medians, etc.
* Suppose we are analyzing salaries for jobs in data science. If our data set contains values that are missing, finding averages, distributions, etc. may result in erroneous conclusions.

Common Data Cleaning Tasks:

* Checking for missing values
* Checking for duplicate entries

Additionally, sometimes datasets are just not in the format that we would prefer:
* They have extra columns we don't need.
* They have column names that are inconvenient to work with.
* They have data in a format other than what we would prefer.

These too, can be considered data cleaning steps.

In the particular case of data that is not precisely in the format that you want it in, you may find it helpful to add new columns computed from existing columns to reformat the data.  We typically use the `apply` function to do this.



## `nan`, `None`, etc.


Some times you'll see values `nan` in datasets, which stands for "Not A Number".  

Sometimes this is a true `nan`, and sometimes it's the string "nan".

Either way, it signifies missing data.

Python value of `None` can also signify missing data.

You need to determine whether your data contains such values, and if so, whether that will impact your analysis, and whether you need to remove the rows that having missing data.



# What's a `nan`?

You'll see lots of references to `nan`.

Let's explore `nan` a bit


In [ ]:
1 + 2 + 3 + np.nan

nan

In [ ]:
np.nan

nan

In [ ]:
1 + np.nan

nan

In [ ]:
1 == np.nan

False

In [ ]:
np.nan == np.nan # Comparing `nan` with == doesn't work!

False

In [ ]:
np.nan is np.nan # Comparing `nan` with `is` does work!

True

In [ ]:
1 is np.nan # This works, but gives us warnings

<>:1: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
<>:1: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
/tmp/ipython-input-2827583387.py:1: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  1 is np.nan # This works, but gives us warnings


False

In [ ]:
np.isnan(1) # But a cleaner way is with the np.isnan(...) function

False

In [ ]:
np.isnan(np.nan)

True

In [ ]:
x = np.nan
x == x

False

In [ ]:
x != x

True

In [ ]:
np.isnan(x)

True

## Data Cleaning Example: Data Science Salaries

Our data folder contains a data set of Data Science Salaries that has some missing values in it.  We'll use this as an exercise in data cleaning.

Let's first look at this table:

In [ ]:
salaries_unclean = Table.read_table(DATA_FOLDER + "data_science_salaries_raw.csv")
salaries_unclean

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
Data Engineer,Full-Time,Senior,Expert,210000,United States Dollar,United States,210000,United States,Medium,2023
Data Engineer,Full-Time,Senior,Expert,165000,United States Dollar,United States,165000,United States,Medium,2023
Data Engineer,Full-Time,Senior,Expert,185900,United States Dollar,United States,185900,United States,Medium,2023
Data Engineer,Full-Time,Senior,Expert,129300,United States Dollar,United States,129300,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,140000,United States Dollar,United States,140000,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,126000,United States Dollar,United States,126000,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,170000,United States Dollar,United States,170000,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,130000,United States Dollar,United States,130000,United States,Medium,2023
Data Engineer,Full-Time,Mid,Intermediate,85000,British Pound Sterling,United Kingdom,104584,United Kingdom,Medium,2023
Data Engineer,Full-Time,Mid,Intermediate,75000,British Pound Sterling,United Kingdom,92280,United Kingdom,Medium,2023


To get to know the data, let's take a few random samples of ten rows.  You may or may not see anything that looks sketchy, but it's a good start.

In [ ]:
salaries_unclean.sample(10)


Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
Head of Data Science,Full-Time,Mid,Intermediate,110000,United States Dollar,United States,110000,United States,Small,2021
Data Scientist,Full-Time,Senior,Expert,159832,United States Dollar,United States,159832,United States,Medium,2023
Analytics Engineer,Full-Time,Senior,Expert,214200,United States Dollar,United States,214200,United States,Medium,2023
Data Analyst,Full-Time,Senior,Expert,175000,United States Dollar,United States,175000,United States,Medium,2022
Data Analyst,Full-Time,Senior,Expert,75000,United States Dollar,United States,75000,United States,Medium,2023
Data Architect,Full-Time,Senior,Expert,345600,United States Dollar,United States,345600,United States,Medium,2022
Analytics Engineer,Full-Time,Senior,Expert,140250,United States Dollar,United States,140250,United States,Medium,2022
Data Science Manager,Full-Time,Senior,Expert,191765,United States Dollar,United States,191765,United States,Medium,2023
Data Engineer,Full-Time,Mid,Intermediate,78000,United States Dollar,Brazil,78000,Brazil,Medium,2022
Data Scientist,Full-Time,Senior,Expert,167500,United States Dollar,United States,167500,United States,Medium,2023


Next, let's check the Salary column, since that's a column that we might be doing analysis on.  We want to see if any of the values in that column have the special value `nan`, which stands for "Not a Number".

Here's how we can check that:


In [ ]:
# Pull out all of the salaries into a column
salaries = salaries_unclean.column('Salary in USD')
salaries

array([ 210000.,  165000.,  185900., ...,  104584.,   92280.,  208500.])

In [ ]:
len(salaries)

3311

In [ ]:
demo = make_array(1, 2, np.nan, 4, 5)
demo

array([  1.,   2.,  nan,   4.,   5.])

In [ ]:
np.isnan(demo)

array([False, False,  True, False, False], dtype=bool)

In [ ]:
np.sum(make_array(True, False, False, True, True))

3

In [ ]:
# Check whether each is the value nan
np.isnan(salaries_unclean.column('Salary in USD'))

array([False, False, False, ..., False, False, False], dtype=bool)

In [ ]:
# Count how many of these are true
np.count_nonzero(np.isnan(salaries_unclean.column('Salary in USD')))

4

So, we could write a function that will take a table and column name, and tell us how many values in that column are the value `nan`:

In [ ]:
def count_nans(table, column_name):
    return np.count_nonzero(np.isnan(table.column(column_name)))

salary_in_USD_nans = count_nans(salaries_unclean, 'Salary in USD')
salary_nans = count_nans(salaries_unclean, 'Salary')
year_nans = count_nans(salaries_unclean, 'Year')
print(f"Number of nans in Salary in USD: {salary_in_USD_nans}")
print(f"Number of nans in Salary: {salary_nans}")
print(f"Number of nans in Year: {year_nans}")

Number of nans in Salary in USD: 4
Number of nans in Salary: 4
Number of nans in Year: 0


Now this tells us how many `nan` values there are, i.e. whether we need to continue with cleaning.  But it doesn't tell us which rows have those values, and it doesn't remove them from the data set.  So we have more work to do.

Here's a hint:

> Hint: In Python, `nan` is equal to *nothing*, not even itself. So, to check if a variable x is `nan`, you can test x != x.

So, we can't directly check whether a variable is equal to `nan`.  Check this out!



In [ ]:
x = np.nan
x

nan

In [ ]:
type(x)

float

In [ ]:
type(np.nan)

float

In [ ]:
np.isnan(x)

True

In [ ]:
x == x

False

In [ ]:
x == np.nan

False

In [ ]:
x != x

True

In [ ]:
np.isnan(salaries_unclean.column('Salary in USD'))

array([False, False, False, ..., False, False, False], dtype=bool)

It appears that testing whether something is a `nan` value can be a bit tricky!  But the `np.isnan` function appears to be a safe way to do it:

In [ ]:
five_rows = salaries_unclean.sample(5)
five_rows

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
Data Analyst,Full-Time,Senior,Expert,136260,United States Dollar,United States,136260,United States,Medium,2022
Data Engineer,Full-Time,Senior,Expert,160000,United States Dollar,United States,160000,United States,Large,2022
Data Analyst,Full-Time,Senior,Expert,149000,United States Dollar,United States,149000,United States,Medium,2022
Data Engineer,Full-Time,Senior,Expert,223250,United States Dollar,United States,223250,United States,Medium,2023
Machine Learning Engineer,Full-Time,Senior,Expert,60000,Euro,Finland,63040,Finland,Small,2022


In [ ]:
five_rows.where(make_array(False, True, False, True, True))

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
Data Engineer,Full-Time,Senior,Expert,160000,United States Dollar,United States,160000,United States,Large,2022
Data Engineer,Full-Time,Senior,Expert,223250,United States Dollar,United States,223250,United States,Medium,2023
Machine Learning Engineer,Full-Time,Senior,Expert,60000,Euro,Finland,63040,Finland,Small,2022


In [ ]:
nan_rows = salaries_unclean.where(np.isnan(salaries_unclean.column('Salary in USD')))
nan_rows

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
nan,Full-Time,nan,Expert,nan,United States Dollar,United States,nan,United States,Medium,2023
nan,Full-Time,nan,Expert,nan,United States Dollar,United States,nan,United States,Medium,2023
nan,Full-Time,nan,Expert,nan,United States Dollar,United States,nan,United States,Medium,2023
nan,Full-Time,nan,Expert,nan,United States Dollar,United States,nan,United States,Medium,2023


Since that worked, let's try the opposite; we'll just put the function `np.logical_not` in front of `np.isnan`:

In [ ]:
valid_rows = salaries_unclean.where(np.logical_not(np.isnan(salaries_unclean.column('Salary in USD'))))
valid_rows

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
Data Engineer,Full-Time,Senior,Expert,210000,United States Dollar,United States,210000,United States,Medium,2023
Data Engineer,Full-Time,Senior,Expert,165000,United States Dollar,United States,165000,United States,Medium,2023
Data Engineer,Full-Time,Senior,Expert,185900,United States Dollar,United States,185900,United States,Medium,2023
Data Engineer,Full-Time,Senior,Expert,129300,United States Dollar,United States,129300,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,140000,United States Dollar,United States,140000,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,126000,United States Dollar,United States,126000,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,170000,United States Dollar,United States,170000,United States,Medium,2023
Data Scientist,Full-Time,Senior,Expert,130000,United States Dollar,United States,130000,United States,Medium,2023
Data Engineer,Full-Time,Mid,Intermediate,85000,British Pound Sterling,United Kingdom,104584,United Kingdom,Medium,2023
Data Engineer,Full-Time,Mid,Intermediate,75000,British Pound Sterling,United Kingdom,92280,United Kingdom,Medium,2023


In [ ]:
valid_rows.num_rows

3307

In [ ]:
nan_rows.num_rows

4

In [ ]:
valid_rows.num_rows + nan_rows.num_rows

3311

In [ ]:
salaries_unclean.num_rows

3311

Let's check whether this data is clean:


In [ ]:
salary_in_USD_nans = count_nans(valid_rows, 'Salary in USD')
salary_nans = count_nans(valid_rows, 'Salary')
year_nans = count_nans(valid_rows, 'Year')
print(f"Number of nans in Salary in USD: {salary_in_USD_nans}")
print(f"Number of nans in Salary: {salary_nans}")
print(f"Number of nans in Year: {year_nans}")

Number of nans in Salary in USD: 0
Number of nans in Salary: 0
Number of nans in Year: 0


It appears we got rid of all of the `nan` values!  But as a sanity check, let's make sure that we have the number of rows we expect.

How many rows were in the original dataset?

In [ ]:
salaries_unclean.num_rows

3311

In [ ]:
nan_rows.num_rows

4

In [ ]:
valid_rows.num_rows

3307

In [ ]:
# Does it add up?
salaries_unclean.num_rows == nan_rows.num_rows + valid_rows.num_rows

True

It appears that this did the trick!  So `valid_rows` is now our cleaned data set, free of any rows that:

* contain `nan` in `Salary` or
* contain `Salary in USD`.

Example of using with_columns:


In [ ]:
t = Table().with_columns({

    'letter': ['a', 'b', 'c', 'z'],

    'count':  [  9,   3,   3,   1],

    'points': [  1,   2,   2,  10],

})
t

letter,count,points
a,9,1
b,3,2
c,3,2
z,1,10


In [ ]:
t.rows

Rows(letter | count | points
a      | 9     | 1
b      | 3     | 2
c      | 3     | 2
z      | 1     | 10)

# Removing Duplicates

Here is some code to remove duplicates.  It's a little tricky, but we'll look through it a bit at a time, and try to understand how it works.


First, we'll use someting called `.rows`, which we can use to turn a table into a numpy array.

It is explained at this link if you want to read more about it:
* <https://www.data8.org/datascience/_autosummary/datascience.tables.Table.rows.html#datascience.tables.Table.rows>

In [ ]:
# Convert the table to a numpy array
# https://www.data8.org/datascience/_autosummary/datascience.tables.Table.rows.html#datascience.tables.Table.rows
# type(salaries_unclean.rows)
data = np.array(valid_rows.rows)
data


array([['Data Engineer', 'Full-Time', 'Senior', ..., 'United States',
        'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Senior', ..., 'United States',
        'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Senior', ..., 'United States',
        'Medium', '2023'],
       ..., 
       ['Data Engineer', 'Full-Time', 'Mid', ..., 'United Kingdom',
        'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Mid', ..., 'United Kingdom',
        'Medium', '2023'],
       ['Analytics Engineer', 'Full-Time', 'Senior', ..., 'United States',
        'Medium', '2023']],
      dtype='<U40')

The variable `data` now contains all of the data in our table, but it's not in table form any more.  We've converted it into a `numpy.ndarray`.  The data is all still there, but represented in a different form.

We can't directly do table methods on it, but we can do `numpy` methods on it, and we are going to use that to look for, and remove duplicates.  Then, we'll turn it back to the `table` format that we know and love.

## Using `np.unique`

The `numpy` function `np.unique` allows us to look through a `numpy` array for duplicates.  It gives us back *two* return values, as shown in this code cell:


In [ ]:
experiment_table = Table().with_columns({

    'letter': ['a', 'a', 'c', 'a'],

    'count':  [  9,   9,   3,   9],

    'points': [  1,   1,   2,  1],

})
experiment_table

letter,count,points
a,9,1
a,9,1
c,3,2
a,9,1


In [ ]:
experiment_data = np.array(experiment_table.rows)
experiment_data

array([['a', '9', '1'],
       ['a', '9', '1'],
       ['c', '3', '2'],
       ['a', '9', '1']],
      dtype='<U21')

In [ ]:
ur_experiment_data, counts_experiment_data = np.unique(experiment_data, axis=0, return_counts=True)

In [ ]:

print("\n\n********* unique rows *********")
print(ur_experiment_data)

print("\n\n********* counts *********")
print(counts_experiment_data)




********* unique rows *********
[['a' '9' '1']
 ['c' '3' '2']]


********* counts *********
[3 1]


In [ ]:
# finds unique elements of an array and also returns the number of times each unique item appears
# axis to operate on, here axis=0 is important if we want to find unique rows
# (axis=1 would give us unique columns - doesn't make sense in this case)
unique_rows, counts =  np.unique(data, axis=0, return_counts=True)# gives us two arrays


In [ ]:

print("\n\n********* unique rows *********")
print(unique_rows)

print("\n\n********* counts *********")
print(counts)




********* unique rows *********
[['AI Architect' 'Full-Time' 'Executive' ..., 'Belgium' 'Large' '2023']
 ['AI Architect' 'Full-Time' 'Senior' ..., 'United States' 'Large' '2023']
 ['AI Architect' 'Full-Time' 'Senior' ..., 'United States' 'Medium' '2023']
 ..., 
 ['Staff Data Scientist' 'Contract' 'Senior' ..., 'United States' 'Medium'
  '2021']
 ['Staff Data Scientist' 'Full-Time' 'Senior' ..., 'United States' 'Medium'
  '2020']
 ['Staff Machine Learning Engineer' 'Full-Time' 'Senior' ...,
  'United States' 'Medium' '2021']]


********* counts *********
[1 1 1 ..., 1 1 1]


In [ ]:
# This code finds the rows that had duplicates among all of the unique rows.
# I'll be honest; I didn't understand how this code worked when I first saw it
# But we can use Gemini AI on Google CoLab to explain it

duplicate_rows = unique_rows[counts > 1]
duplicate_rows

array([['Analytics Engineer', 'Full-Time', 'Senior', 'Expert', '208500.0',
        'United States Dollar', 'United States', '208500.0',
        'United States', 'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Mid', 'Intermediate', '75000.0',
        'British Pound Sterling', 'United Kingdom', '92280.0',
        'United Kingdom', 'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Mid', 'Intermediate', '85000.0',
        'British Pound Sterling', 'United Kingdom', '104584.0',
        'United Kingdom', 'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Senior', 'Expert', '165000.0',
        'United States Dollar', 'United States', '165000.0',
        'United States', 'Medium', '2023'],
       ['Data Engineer', 'Full-Time', 'Senior', 'Expert', '210000.0',
        'United States Dollar', 'United States', '210000.0',
        'United States', 'Medium', '2023'],
       ['Data Scientist', 'Full-Time', 'Senior', 'Expert', '130000.0',
        'United States Dollar', 'Unite

# Making sure everything is accounted for

We originally had this many rows in our dataset before we started removing duplicates:


In [ ]:
valid_rows.num_rows

3307

We now have this many rows in the unique_rows (note that we use `len` because this is a numpy array, not yet a table):

In [ ]:
len(unique_rows)

3300

That means we removed seven duplicates.   Let's look at the counts array, and find how many of the counts were greater than 1:

In [ ]:
counts[counts > 1]

array([2, 2, 2, 2, 2, 2, 2])

That looks like seven elements, i.e. seven rows that had a count greater than 1 that were removed.  Each of then appeared twice (our math would be a bit trickier if any of these numbers were bigger than 2).

In [ ]:
len(counts[counts>1])

7

In [ ]:
# Do these match?
# Does what we started with == what we ended up with  + what we removed ?
valid_rows.num_rows == len(unique_rows) + len(counts[counts>1])


True

Yes it does! So all rows are accounted for, and `unique_rows` contains the de-duplicated data.

## Putting the table back together

Now that we have the `unique_rows`, we can put the table back together again.  

First, let's go back to our source of data, which was `valid_rows` and get all of the labels:

In [ ]:
unique_rows

array([['AI Architect', 'Full-Time', 'Executive', ..., 'Belgium', 'Large',
        '2023'],
       ['AI Architect', 'Full-Time', 'Senior', ..., 'United States',
        'Large', '2023'],
       ['AI Architect', 'Full-Time', 'Senior', ..., 'United States',
        'Medium', '2023'],
       ..., 
       ['Staff Data Scientist', 'Contract', 'Senior', ..., 'United States',
        'Medium', '2021'],
       ['Staff Data Scientist', 'Full-Time', 'Senior', ...,
        'United States', 'Medium', '2020'],
       ['Staff Machine Learning Engineer', 'Full-Time', 'Senior', ...,
        'United States', 'Medium', '2021']],
      dtype='<U40')

In [ ]:
# Column names
valid_rows.labels

('Job Title',
 'Employment Type',
 'Experience Level',
 'Expertise Level',
 'Salary',
 'Salary Currency',
 'Company Location',
 'Salary in USD',
 'Employee Residence',
 'Company Size',
 'Year')

In [ ]:
# Putting together a table with those labels and the unique rows
filtered_unique_salaries = Table(valid_rows.labels).with_rows(unique_rows)
filtered_unique_salaries

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
AI Architect,Full-Time,Executive,Director,200000.0,Euro,United States,215936.0,Belgium,Large,2023
AI Architect,Full-Time,Senior,Expert,200000.0,United States Dollar,United States,200000.0,United States,Large,2023
AI Architect,Full-Time,Senior,Expert,204000.0,United States Dollar,United States,204000.0,United States,Medium,2023
AI Architect,Full-Time,Senior,Expert,330000.0,United States Dollar,United States,330000.0,United States,Medium,2023
AI Developer,Full-Time,Entry,Junior,120000.0,United States Dollar,Bosnia and Herzegovina,120000.0,Bosnia and Herzegovina,Small,2023
AI Developer,Full-Time,Entry,Junior,200000.0,Euro,Germany,215936.0,Germany,Large,2023
AI Developer,Full-Time,Entry,Junior,60000.0,Euro,Germany,64781.0,Germany,Medium,2023
AI Developer,Full-Time,Entry,Junior,80000.0,United States Dollar,Sweden,80000.0,Sweden,Medium,2023
AI Developer,Full-Time,Mid,Intermediate,100000.0,Singapore Dollar,Finland,74883.0,Finland,Medium,2023
AI Developer,Full-Time,Mid,Intermediate,200000.0,United States Dollar,United States,200000.0,United States,Medium,2023


Let's apply the same technique as before to verify that there are now no duplicates.


In [ ]:
# finds unique elements of an array and also returns the number of times each unique item appears
# axis to operate on, here axis=0 is important if we want to find unique rows
# (axis=1 would give us unique columns - doesn't make sense in this case)

fus_data = np.array(filtered_unique_salaries.rows)

unique_rows_fs, counts_fs = np.unique(fus_data, axis=0, return_counts=True) # gives us two arrays


In [ ]:

print("\n\n********* unique rows fs *********")
print(unique_rows_fs)

print("\n\n********* counts fs *********")
print(counts_fs)





********* unique rows fs *********
[['AI Architect' 'Full-Time' 'Executive' ..., 'Belgium' 'Large' '2023']
 ['AI Architect' 'Full-Time' 'Senior' ..., 'United States' 'Large' '2023']
 ['AI Architect' 'Full-Time' 'Senior' ..., 'United States' 'Medium' '2023']
 ..., 
 ['Staff Data Scientist' 'Contract' 'Senior' ..., 'United States' 'Medium'
  '2021']
 ['Staff Data Scientist' 'Full-Time' 'Senior' ..., 'United States' 'Medium'
  '2020']
 ['Staff Machine Learning Engineer' 'Full-Time' 'Senior' ...,
  'United States' 'Medium' '2021']]


********* counts fs *********
[1 1 1 ..., 1 1 1]


In [ ]:
duplicate_rows_fs = unique_rows_fs[counts_fs > 1]
duplicate_rows_fs

array([], shape=(0, 11),
      dtype='<U40')

We see that the duplicate_rows_fs variable is an empty array, so we can now be confident there are no duplicates in our data set.  The data set is clean!

In [ ]:
filtered_unique_salaries

Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
AI Architect,Full-Time,Executive,Director,200000.0,Euro,United States,215936.0,Belgium,Large,2023
AI Architect,Full-Time,Senior,Expert,200000.0,United States Dollar,United States,200000.0,United States,Large,2023
AI Architect,Full-Time,Senior,Expert,204000.0,United States Dollar,United States,204000.0,United States,Medium,2023
AI Architect,Full-Time,Senior,Expert,330000.0,United States Dollar,United States,330000.0,United States,Medium,2023
AI Developer,Full-Time,Entry,Junior,120000.0,United States Dollar,Bosnia and Herzegovina,120000.0,Bosnia and Herzegovina,Small,2023
AI Developer,Full-Time,Entry,Junior,200000.0,Euro,Germany,215936.0,Germany,Large,2023
AI Developer,Full-Time,Entry,Junior,60000.0,Euro,Germany,64781.0,Germany,Medium,2023
AI Developer,Full-Time,Entry,Junior,80000.0,United States Dollar,Sweden,80000.0,Sweden,Medium,2023
AI Developer,Full-Time,Mid,Intermediate,100000.0,Singapore Dollar,Finland,74883.0,Finland,Medium,2023
AI Developer,Full-Time,Mid,Intermediate,200000.0,United States Dollar,United States,200000.0,United States,Medium,2023


## Some other checks we could perform

There are other checks we could perform when cleaning data:

* We could verify that all values in a field are within a reasonable range, by taking the min and max of a column, and verifying that the values seem reasonable.

* We could convert certain categories/values to upper or lower case - For example, if the data had the "Expert" level with several cases such as "EXpert", "EXPERT", "expert".  We might benefit from converting them all to lower of upper cased versions.  This might involve writing a function, using apply to create a new column, and then dropping and renaming the old column.



## Asking ChatGPT about `unique_rows[counts > 1]`

We asked ChatGPT:

```
When using the datascience library and the numpy library, what does this expression do:
  unique_rows[counts > 1]

```

Here's what we got back:

---

The expression `unique_rows[counts > 1]` filters the unique_rows array to only include rows that have a count greater than 1.

To provide more context, `np.unique` returns the unique rows of a given array along with their counts. The result is returned as two arrays: unique_rows contains the unique rows, and counts contains the corresponding counts for each unique row.

By using the boolean condition `counts > 1`, we create a boolean array where each element represents whether the count for the corresponding row is greater than 1.

By applying this boolean array as an index to `unique_rows`, we only select the rows where the count is greater than 1, effectively filtering out the unique rows and keeping duplicate rows.

Here is an example to demonstrate this:

```
import numpy as np

# Create an array with duplicate rows
arr = np.array([[1, 2], [2, 3], [1, 2], [3, 4], [2, 3]])

# Get the unique rows and their counts
unique_rows, counts = np.unique(arr, axis=0, return_counts=True)

# Filter unique rows to only include rows with count > 1
filtered_rows = unique_rows[counts > 1]

# Display the filtered rows
print(filtered_rows)
```

Output:
```
[[1 2]
 [2 3]]
```
The resulting filtered_rows array contains the rows `[1, 2]` and `[2, 3]`, which are the duplicate rows in the original array.
